In [53]:
import google.generativeai as genai
from collections import defaultdict
import json

In [ ]:
genai.configure(api_key = "")

products_file = 'products.json'
categories_file = 'categories.json'
delimiter = "####"

In [42]:
aimodel   = "gemini-3.5-flash-lite"

In [43]:
def get_completion_from_messages(messages, model=aimodel, temperature=0,max_output_tokens=10000):
    model = genai.GenerativeModel(model)
    response = model.generate_content(messages,generation_config={"temperature": 0,"max_output_tokens": max_output_tokens})
    return response.text

In [44]:
def find_category_and_product_only(user_input, products_and_category):
    delimiter = "####"
    system_message = """
    You will be provided with customer service queries. The customer service query will be delimited with {delimiter} characters.
    Output a python list of objects, where each object has the following format:
    "category": <one of Computers and Laptops, Smartphones and Accessories, Televisions and Home Theater Systems, 
    Gaming Consoles and Accessories, Audio Equipment, Cameras and Camcorders>,
    OR
    "products": <a list of products that must be found in the allowed products below>

    Where the categories and products must be found in the customer service query.
    If a product is mentioned, it must be associated with the correct category in the allowed products list below.
    If no products or categories are found, output an empty list. Return in json format

    Allowed products: 
    Computers and Laptops category:
    TechPro Ultrabook,
    BlueWave Gaming Laptop,
    PowerLite Convertible,
    TechPro Desktop,
    BlueWave Chromebook

    Smartphones and Accessories category:
    SmartX ProPhone,
    MobiTech PowerCase,
    SmartX MiniPhone,
    MobiTech Wireless Charger,
    SmartX EarBuds

    Televisions and Home Theater Systems category:
    CineView 4K TV,
    SoundMax Home Theater,
    CineView 8K TV,
    SoundMax Soundbar,
    CineView OLED TV

    Gaming Consoles and Accessories category:
    GameSphere X,
    ProGamer Controller,
    GameSphere Y,
    ProGamer Racing Wheel,
    GameSphere VR Headset

    Audio Equipment category:
    AudioPhonic Noise-Canceling Headphones,
    WaveSound Bluetooth Speaker,
    AudioPhonic True Wireless Earbuds,
    WaveSound Soundbar,
    AudioPhonic Turntable

    Cameras and Camcorders category:
    FotoSnap DSLR Camera,
    ActionCam 4K,
    FotoSnap Mirrorless Camera,
    ZoomMaster Camcorder,
    FotoSnap Instant Camera
    
    Only output the list of objects, nothing else.
    """
    messages = [  
        {"role": "system", "content": system_message},    
        {"role": "user",   "content": f"{delimiter}{user_input}{delimiter}"}
    ] 
    return get_completion_from_messages(messages)

In [45]:
def get_completion_from_messages(messages, model="gemini-1.5-pro-latest", temperature=0.7, max_tokens=500):
    
    client = genai.GenerativeModel(model_name = aimodel)  # Initialize Gemini model

    # Remove "system" role and merge instructions into user's first message
    formatted_messages = []
    system_message = ""

    for message in messages:
        if message["role"] == "system":
            system_message = message["content"]  # Store system instruction
        elif "content" in message:  
            formatted_messages.append({
                "role": message["role"],  
                "parts": [{"text": message["content"]}]  # Correct format for Gemini
            })
        else:
            raise ValueError(f"Message missing 'content' field: {message}")  # Debugging

    # If there was a system message, merge it into the first user message
    if formatted_messages and system_message:
        formatted_messages[0]["parts"][0]["text"] = system_message + "\n\n" + formatted_messages[0]["parts"][0]["text"]

    response = client.generate_content(
        formatted_messages,  
        generation_config={
            "temperature": temperature,
            "max_output_tokens": max_tokens
        }
    )

    return response.text

In [46]:
def get_products_and_category():
    """
    Retrieves product details and categorizes them properly.
    Ensures all categories exist even if they have no products.
    """
    predefined_categories = [
        "Computers and Laptops", "Smartphones and Accessories",
        "Televisions and Home Theater Systems", "Gaming Consoles and Accessories",
        "Audio Equipment", "Cameras and Camcorders"
    ]

    try:
        products = get_products()  # Ensure this function is defined and returns a dictionary
        if not isinstance(products, dict):
            raise ValueError("get_products() did not return a valid dictionary.")

        products_by_category = defaultdict(list)

        # Add existing products to their categories
        for product_name, product_info in products.items():
            category = product_info.get('category', None)
            if category:
                products_by_category[category].append(product_name)

        # Ensure all predefined categories exist (even if empty)
        for category in predefined_categories:
            if category not in products_by_category:
                products_by_category[category] = []

        return dict(products_by_category)

    except Exception as e:
        print(f"Error in get_products_and_category: {e}")
        return {category: [] for category in predefined_categories}  # Return empty categories

In [47]:
def read_string_to_list(input_data):
    """
    Converts a JSON string to a Python list.
    Handles cases where the input is already a list or may be malformed.
    """
    if isinstance(input_data, list):  
        return input_data  # Already a list, no need to process

    if not input_data:
        return []

    try:
        # Ensure input is a string before processing
        cleaned_input = str(input_data).strip().strip("```").replace("json", "").strip()
        return json.loads(cleaned_input)
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON string - {e}")
        return []

In [48]:
def generate_output_string(data_list):
    output_string = ""

    if not data_list:
        return output_string

    for data in data_list:
        try:
            if "products" in data:
                for product_name in data["products"]:
                    product = get_product_by_name(product_name)
                    if product:
                        output_string += json.dumps(product, indent=4) + "\n"
                    else:
                        print(f"Error: Product '{product_name}' not found")
            elif "category" in data:
                category_products = get_products_by_category(data["category"])
                for product in category_products:
                    output_string += json.dumps(product, indent=4) + "\n"
            else:
                print("Error: Invalid object format")
        except Exception as e:
            print(f"Error: {e}")

    return output_string

In [49]:
# Product lookup (either by category or by product within category)
def get_product_by_name(name):
    products = get_products()
    return products.get(name, None)

def get_products_by_category(category):
    products = get_products()
    return [product for product in products.values() if product.get("category") == category]

def get_mentioned_product_info(data_list):
    """
    Used in L5 and L6
    """
    product_info_l = []

    if not data_list:
        return product_info_l

    for data in data_list:
        try:
            if "products" in data:
                for product_name in data["products"]:
                    product = get_product_by_name(product_name)
                    if product:
                        product_info_l.append(product)
                    else:
                        print(f"Error: Product '{product_name}' not found")
            elif "category" in data:
                category_products = get_products_by_category(data["category"])
                product_info_l.extend(category_products)
            else:
                print("Error: Invalid object format")
        except Exception as e:
            print(f"Error: {e}")

    return product_info_l

In [50]:
def get_products():
    with open(products_file, 'r') as file:
        products = json.load(file)
    return products

In [51]:
def get_products():
    with open(products_file, 'r') as file:
        products = json.load(file)
    return products

In [54]:
#Setting up a full flow
user_input = "tell me about the SmartX Pro Phone and the FotoSnap Camera, the DSLR one. Also, what tell me about your TVs?"


#Step 1 : Validate input - moderate the input data

#Step 2 : Extract products and categories
category_and_product_response = find_category_and_product_only(user_input, get_products_and_category())
print(category_and_product_response)

category_and_product_list = read_string_to_list(category_and_product_response)
print(category_and_product_list)


# Step 3: Fetch product information
#product_information = generate_output_string(category_and_product_list)
#print(product_information)


# Step 4: Generate customer service response
system_message_content_4 = "Be concise. Only include product description and key features. Do NOT include price, rating, or extra explanation."

messages = [
        { "role": "user", "content": system_message_content_4 + f"\n\n{delimiter}{user_input}{delimiter}", },  # Merged system message
        {"role": "user", "content": f"Relevant product information:\n{product_information}"},
           ]

final_response = get_completion_from_messages(messages)

all_messages = []
all_messages.append({"role": "user", "content": final_response})



# Step 5: Evaluate response quality
step_6_system_message_content = """
You are an evaluator.

Check if the assistant response answers the user's question.
    
Rules:
    - If ALL requested products are answered → Y
    - If ANY product is missing → N
    - Ignore wording differences
    - Be lenient
    
Return ONLY:
Y or N
    """

evaluation_messages = [
        {
            "role": "user",
            "content": step_6_system_message_content
            + f"\n\nCustomer message: {delimiter}{user_input}{delimiter}\n"
            + f"Agent response: {delimiter}{final_response}{delimiter}\n"
            + f"Does the response sufficiently answer the question? reply in single word Y or N",
        }
    ]

evaluation_response = get_completion_from_messages(evaluation_messages)

# Ensure strict formatting
cleaned_evaluation_response = evaluation_response.strip().upper()  # Remove spaces & enforce uppercase


print(f"Step 6: Evaluation result: {cleaned_evaluation_response}")

# Step 7: Decision based on evaluation
if cleaned_evaluation_response == "Y":
    print("Step 7: Response is approved.")
    #return final_response, all_messages
else:
    print("Step 7: Evaluation failed, but returning response anyway.")

print( "I'm unable to provide the information you're looking for. Let me connect you with a representative.", all_messages)


[
  {
    "category": "Smartphones and Accessories",
    "products": ["SmartX ProPhone"]
  },
  {
    "category": "Cameras and Camcorders",
    "products": ["FotoSnap DSLR Camera"]
  },
  {
    "category": "Televisions and Home Theater Systems"
  }
]
[{'category': 'Smartphones and Accessories', 'products': ['SmartX ProPhone']}, {'category': 'Cameras and Camcorders', 'products': ['FotoSnap DSLR Camera']}, {'category': 'Televisions and Home Theater Systems'}]
Step 6: Evaluation result: Y
Step 7: Response is approved.
I'm unable to provide the information you're looking for. Let me connect you with a representative. [{'role': 'user', 'content': '**SmartX Pro Phone**\n*Product Description:* A high-performance flagship smartphone designed for productivity, creativity, and seamless everyday use.\n*Key Features:*\n* 6.8-inch Dynamic AMOLED 120Hz display\n* Octa-core processor with 12GB RAM\n* Triple-lens 108MP camera system with 8K video recording\n* All-day 5,000mAh battery with 65W fast cha